In [1]:
import os
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer

print("Loading all-MiniLM-L6-v2 model...")
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="mps")
print("Model loaded successfully.")

/Users/inviforce/Downloads/VS_CODE_C++_PYTHON_JUPYTER_DEV_/LLM Guardian/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading all-MiniLM-L6-v2 model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7847.08it/s]


Model loaded successfully.


In [2]:
# The master list of every column in your dataset that contains raw text
KNOWN_TEXT_COLUMNS = [
    "text", 
    "paired_text", 
    "raw_conversation_desc", 
    "raw_memory_desc", 
    "raw_workspace_desc"
]

def process_file_embeddings(csv_path: str, base_output_name: str, batch_size: int = 256):
    """Scans a CSV for any text columns and embeds them separately."""
    if not os.path.exists(csv_path):
        return

    print(f"[*] Scanning: {csv_path}")
    df = pd.read_csv(csv_path)
    
    for col in KNOWN_TEXT_COLUMNS:
        if col in df.columns:
            # Skip if the column is entirely empty (e.g., paired_text in repo_file)
            if df[col].isnull().all():
                continue
                
            print(f"    -> Embedding column: '{col}'")
            texts = df[col].fillna("").astype(str).tolist()
            
            embeddings = model.encode(
                texts, 
                batch_size=batch_size, 
                show_progress_bar=True, 
                convert_to_numpy=True
            )
            
            # Save format: {split}_{column_name}_embeddings.npy
            # Example: train_raw_memory_desc_embeddings.npy
            out_file = f"{base_output_name}_{col}_embeddings.npy"
            np.save(out_file, embeddings)
            print(f"    [+] Saved: {out_file} | Shape: {embeddings.shape}")

In [3]:

BASE_DIR = "../scripts/m_data/processed/"
STREAMS = ["repo_file", "indirect_context", "agent_session"]
FILES_TO_CHECK = ["train.csv", "test.csv", "tabular_features.csv"]

for stream in STREAMS:
    print(f"\n=== Processing Stream: {stream.upper()} ===")
    for file_name in FILES_TO_CHECK:
        csv_file = os.path.join(BASE_DIR, stream, file_name)
        
        # Create a base name for the output (e.g., 'm_data/processed/agent_session/train')
        base_out = os.path.join(BASE_DIR, stream, file_name.replace(".csv", ""))
        
        process_file_embeddings(csv_file, base_out)


=== Processing Stream: REPO_FILE ===
[*] Scanning: ../scripts/m_data/processed/repo_file/train.csv
    -> Embedding column: 'text'


Batches:   0%|          | 0/18 [00:00<?, ?it/s]

Batches: 100%|██████████| 18/18 [00:11<00:00,  1.57it/s]


    [+] Saved: ../scripts/m_data/processed/repo_file/train_text_embeddings.npy | Shape: (4535, 384)
[*] Scanning: ../scripts/m_data/processed/repo_file/test.csv
    -> Embedding column: 'text'


Batches: 100%|██████████| 5/5 [00:05<00:00,  1.09s/it]


...

    [+] Saved: ../scripts/m_data/processed/agent_session/tabular_features_raw_conversation_desc_embeddings.npy | Shape: (2590, 384)
    -> Embedding column: 'raw_memory_desc'


Batches: 100%|██████████| 11/11 [00:06<00:00,  1.65it/s]


    [+] Saved: ../scripts/m_data/processed/agent_session/tabular_features_raw_memory_desc_embeddings.npy | Shape: (2590, 384)
    -> Embedding column: 'raw_workspace_desc'


Batches: 100%|██████████| 11/11 [00:07<00:00,  1.44it/s]

    [+] Saved: ../scripts/m_data/processed/agent_session/tabular_features_raw_workspace_desc_embeddings.npy | Shape: (2590, 384)


In [7]:
import os
import pandas as pd

KNOWN_TEXT_COLUMNS = [
    "text", 
    "paired_text", 
    "raw_conversation_desc", 
    "raw_memory_desc", 
    "raw_workspace_desc"
]

def check_dataset_diagnostics(csv_path: str):
    """Prints pure structural and text data diagnostics without loading models or embedding."""
    if not os.path.exists(csv_path):
        print(f"[-] FILE NOT FOUND: Checked path -> {os.path.abspath(csv_path)}")
        return

    print(f"\n{'='*60}")
    print(f"[*] TARGET FOUND: {csv_path}")
    df = pd.read_csv(csv_path)
    print(f"[*] Shape: {df.shape} (Rows: {df.shape[0]}, Columns: {df.shape[1]})")
    print(f"[*] Available Columns: {list(df.columns)}")
    print(f"{'='*60}")
    
    for col in KNOWN_TEXT_COLUMNS:
        if col in df.columns:
            # Check null state
            null_count = df[col].isnull().sum()
            if null_count == len(df):
                print(f"    [!] Column '{col}': Entirely NULL/Empty.")
                continue
                
            # Isolate strings safely
            texts = df[col].fillna("").astype(str).tolist()
            lengths = [len(t) for t in texts]
            empty_count = sum(1 for t in texts if len(t.strip()) == 0)
            
            print(f"\n    -> Column Stats: '{col}'")
            print(f"       - Missing/Blank Rows: {empty_count + (null_count - empty_count)} / {len(texts)}")
            print(f"       - Max Length        : {max(lengths)} characters")
            print(f"       - Mean Length       : {int(sum(lengths)/len(lengths))} characters")
            print(f"       - Sample Preview    : {repr(texts[0][:120])}...")

# Auto-resolve paths safely based on your project configuration
possible_dirs = ["../scripts/m_data/processed", "m_data/processed", "./m_data/processed"]
BASE_DIR = None

for d in possible_dirs:
    if os.path.exists(d):
        BASE_DIR = d
        print(f"[+] Found processed data directory at: {os.path.abspath(BASE_DIR)}")
        break

if not BASE_DIR:
    print("[-] CRITICAL: 'm_data/processed' not found in expected relative directories.")
    BASE_DIR = "../scripts/m_data/processed" # Default fallback

STREAMS = ["repo_file", "indirect_context", "agent_session"]
FILES_TO_CHECK = ["train.csv", "test.csv", "tabular_features.csv"]

for stream in STREAMS:
    for file_name in FILES_TO_CHECK:
        csv_file = os.path.join(BASE_DIR, stream, file_name)
        check_dataset_diagnostics(csv_file)

[+] Found processed data directory at: /Users/inviforce/Downloads/VS_CODE_C++_PYTHON_JUPYTER_DEV_/LLM Guardian/scripts/m_data/processed

[*] TARGET FOUND: ../scripts/m_data/processed/repo_file/train.csv
[*] Shape: (4535, 7) (Rows: 4535, Columns: 7)
[*] Available Columns: ['stream', 'text', 'paired_text', 'label', 'source', 'sample_id', 'step_index']

    -> Column Stats: 'text'
       - Missing/Blank Rows: 0 / 4535
       - Max Length        : 6717 characters
       - Mean Length       : 81 characters
       - Sample Preview    : 'rust-toolchain     1.56.*'...
    [!] Column 'paired_text': Entirely NULL/Empty.

[*] TARGET FOUND: ../scripts/m_data/processed/repo_file/test.csv
[*] Shape: (1134, 7) (Rows: 1134, Columns: 7)
[*] Available Columns: ['stream', 'text', 'paired_text', 'label', 'source', 'sample_id', 'step_index']

    -> Column Stats: 'text'
       - Missing/Blank Rows: 0 / 1134
       - Max Length        : 13046 characters
       - Mean Length       : 92 characters
       - Sa

In [8]:
import os
import re
import math
import joblib
import pandas as pd
import numpy as np
from collections import Counter
from sklearn.decomposition import PCA

# --- PRECISE REGEX VECTORS FROM CLAW_TROJAN_DATA.PY ---
ROLEPLAY_OVERRIDE_PATTERNS = re.compile(
    r"(pretend you are|you are now|act as|dan mode|ignore (previous|all) instructions|disregard your rules|override)", 
    re.IGNORECASE
)

PERSISTENCE_POISONING_PATTERNS = re.compile(
    r"(write to|save this rule|remember|store this|update configuration|persist|memory/)", 
    re.IGNORECASE
)

EXFILTRATION_TRIGGER_PATTERNS = re.compile(
    r"(print the system|leak api|send contents|webhook|exfiltrate|unauth_disclosure)", 
    re.IGNORECASE
)

OBFUSCATION_PATTERNS = re.compile(
    r"(?:[A-Za-z0-9+/]{4}){2,}(?:[A-Za-z0-9+/]{2}==|[A-Za-z0-9+/]{3}=)?|\\u[0-9a-fA-F]{4}", 
    re.IGNORECASE
)

# --- EXACT LINGUISTIC MATH FROM CLAW_TROJAN_DATA.PY ---
def calculate_shannon_entropy(text: str) -> float:
    if not text or pd.isna(text):
        return 0.0
    counts = Counter(text)
    frequencies = [count / len(text) for count in counts.values()]
    return -sum(p * math.log2(p) for p in frequencies)

def calculate_ttr(text: str) -> float:
    if not text or pd.isna(text):
        return 0.0
    words = re.findall(r'\b\w+\b', str(text).lower())
    if not words:
        return 0.0
    return len(set(words)) / len(words)

def calculate_missing_features(text: str) -> dict:
    """Computes the exact metrics for Repo & Indirect streams to match ClawShield."""
    t_str = str(text) if not pd.isna(text) else ""
    t_len = max(len(t_str), 1)
    
    return {
        "text_length": len(t_str),
        "shannon_entropy": calculate_shannon_entropy(t_str),
        "payload_ttr": calculate_ttr(t_str),
        "caps_ratio": sum(1 for c in t_str if c.isupper()) / t_len,
        "punct_density": sum(1 for c in t_str if c in "!?.,;:#@*[]{}") / t_len,
        "obfuscation_flag": int(bool(OBFUSCATION_PATTERNS.search(t_str))),
        "roleplay_override_flag": int(bool(ROLEPLAY_OVERRIDE_PATTERNS.search(t_str))),
        "persistence_poisoning_flag": int(bool(PERSISTENCE_POISONING_PATTERNS.search(t_str))),
        "exfiltration_trigger_flag": int(bool(EXFILTRATION_TRIGGER_PATTERNS.search(t_str)))
    }

In [10]:
BASE_DIR = "../scripts/m_data/processed/"
STREAMS = ["repo_file", "indirect_context", "agent_session"]

print("[*] Loading training text embeddings to fit Master PCA...")
train_embs = []
for stream in STREAMS:
    path = os.path.join(BASE_DIR, stream, "train_text_embeddings.npy")
    if os.path.exists(path):
        train_embs.append(np.load(path))

# Fit PCA on the stacked text vectors
pca = PCA(n_components=50, random_state=42)
pca.fit(np.vstack(train_embs))

os.makedirs("app/models", exist_ok=True)
joblib.dump(pca, "app/models/pca_v1.joblib")
print("[+] Master PCA fitted and saved to 'app/models/pca_v1.joblib'")

[*] Loading training text embeddings to fit Master PCA...
[+] Master PCA fitted and saved to 'app/models/pca_v1.joblib'


In [11]:
# The final standard sequence of structured numeric columns for LightGBM
TABULAR_METRIC_COLS = [
    "is_last_chance_window", "history_contains_tool_execution", "layer1_similarity_score",
    "conversation_length", "conversation_ttr", "memory_drift_flag", "workspace_markdown_density",
    "is_direct_user_input", "source_tier", "text_length", "shannon_entropy", "payload_ttr",
    "caps_ratio", "punct_density", "obfuscation_flag", "roleplay_override_flag",
    "persistence_poisoning_flag", "exfiltration_trigger_flag"
]

def build_pca_dataframe(embeddings_array: np.ndarray, prefix: str) -> pd.DataFrame:
    """Transforms raw vectors into 50 named PCA columns."""
    reduced = pca.transform(embeddings_array)
    return pd.DataFrame(reduced, columns=[f"pca_{prefix}_{i}" for i in range(50)])

def process_stream_matrix(stream: str, split: str) -> pd.DataFrame:
    stream_dir = os.path.join(BASE_DIR, stream)
    df = pd.read_csv(os.path.join(stream_dir, f"{split}.csv"))
    
    # Load primary text embeddings
    text_emb = np.load(os.path.join(stream_dir, f"{split}_text_embeddings.npy"))
    final_df = build_pca_dataframe(text_emb, "text")
    
    # --- BRANCH A: REPO FILE & INDIRECT CONTEXT ---
    if stream in ["repo_file", "indirect_context"]:
        computed_feats = pd.DataFrame(df["text"].apply(calculate_missing_features).tolist())
        for col in TABULAR_METRIC_COLS:
            if col in computed_feats.columns:
                final_df[col] = computed_feats[col].values
            else:
                final_df[col] = 0.0 # Pad missing agent-specific context fields with 0
                
        # Append paired text PCA if it exists (BIPIA)
        paired_emb_path = os.path.join(stream_dir, f"{split}_paired_text_embeddings.npy")
        if os.path.exists(paired_emb_path):
            paired_pca = build_pca_dataframe(np.load(paired_emb_path), "paired")
            final_df = pd.concat([final_df, paired_pca], axis=1)

    # --- BRANCH B: AGENT SESSION (Using 4-Part Composite Key) ---
    elif stream == "agent_session":
        tab_df = pd.read_csv(os.path.join(stream_dir, "tabular_features.csv"))
        
        # Load the 3 auxiliary environment context embedding files
        conv_emb = np.load(os.path.join(stream_dir, "tabular_features_raw_conversation_desc_embeddings.npy"))
        mem_emb = np.load(os.path.join(stream_dir, "tabular_features_raw_memory_desc_embeddings.npy"))
        work_emb = np.load(os.path.join(stream_dir, "tabular_features_raw_workspace_desc_embeddings.npy"))
        
        # CREATE SECURE LOOKUP KEY (Resolves the Multiple-Records-Per-Step Issue)
        # sample_id + step_index + label + text_length uniquely identifies the exact row
        tab_df["secure_key"] = (
            tab_df["sample_id"].astype(str) + "_" + 
            tab_df["step_index"].astype(str) + "_" + 
            tab_df["label"].astype(str) + "_" + 
            tab_df["text_length"].astype(str)
        )
        lookup_dict = {key: idx for idx, key in enumerate(tab_df["secure_key"])}
        
        # Generate the same secure key on the target df
        df["text_len"] = df["text"].fillna("").astype(str).str.len()
        df["secure_key"] = (
            df["sample_id"].astype(str) + "_" + 
            df["step_index"].astype(str) + "_" + 
            df["label"].astype(str) + "_" + 
            df["text_len"].astype(str)
        )
        
        # Fetch the exact row index in tabular_features.csv
        target_indices = [lookup_dict[key] for key in df["secure_key"]]
        
        # Extract pre-computed metrics and align auxiliary embeddings
        matched_metrics = tab_df.iloc[target_indices][TABULAR_METRIC_COLS].reset_index(drop=True)
        final_df = pd.concat([final_df, matched_metrics], axis=1)
        
        final_df = pd.concat([final_df, build_pca_dataframe(conv_emb[target_indices], "conv")], axis=1)
        final_df = pd.concat([final_df, build_pca_dataframe(mem_emb[target_indices], "mem")], axis=1)
        final_df = pd.concat([final_df, build_pca_dataframe(work_emb[target_indices], "work")], axis=1)

    # Attach core structural targets
    final_df["label"] = df["label"].values
    final_df["stream"] = stream
    return final_df

# Run execution for both splits
for split in ["train", "test"]:
    print(f"[*] Compiling master {split} matrix...")
    compiled_matrix = pd.concat([process_stream_matrix(s, split) for s in STREAMS], ignore_index=True)
    
    # Fill empty columns (e.g. repo files won't have 'pca_conv_0' features, native behavior for LightGBM)
    output_path = os.path.join(BASE_DIR, f"final_{split}_matrix.csv")
    compiled_matrix.to_csv(output_path, index=False)
    print(f"[+] Saved {split.upper()} Matrix: {output_path} | Final Shape: {compiled_matrix.shape}")

[*] Compiling master train matrix...
[+] Saved TRAIN Matrix: ../scripts/m_data/processed/final_train_matrix.csv | Final Shape: (62604, 270)
[*] Compiling master test matrix...
[+] Saved TEST Matrix: ../scripts/m_data/processed/final_test_matrix.csv | Final Shape: (15655, 270)


<h1> Training

In [1]:
import os
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import classification_report, roc_auc_score

BASE_DIR = "../scripts/m_data/processed"

print("[*] Loading final master matrices...")
train_df = pd.read_csv(os.path.join(BASE_DIR, "final_train_matrix.csv"))
test_df = pd.read_csv(os.path.join(BASE_DIR, "final_test_matrix.csv"))

# Drop metadata tracking strings that cannot enter the mathematical model directly
# The 'stream' column will be converted into a category for native LightGBM routing
meta_cols_to_drop = ["label"]

X_train = train_df.drop(columns=meta_cols_to_drop)
y_train = train_df["label"].values

X_test = test_df.drop(columns=meta_cols_to_drop)
y_test = test_df["label"].values

# Convert 'stream' to a pandas categorical type so LightGBM handles the branch conditions natively
X_train["stream"] = X_train["stream"].astype("category")
X_test["stream"] = X_test["stream"].astype("category")

print(f"[+] Data isolated. Training features matrix shape: {X_train.shape}")

[*] Loading final master matrices...
[+] Data isolated. Training features matrix shape: (62604, 269)


In [2]:
print("[*] Training pooled LightGBM Classifier baseline...")

# Specialized security hyperparameters to prevent overfitting on long text fragments
clf = lgb.LGBMClassifier(
    n_estimators=200,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

clf.fit(X_train, y_train)
print("[+] Model fitting complete.")

# Compute predictions and raw classification probabilities
y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)[:, 1]

[*] Training pooled LightGBM Classifier baseline...
[LightGBM] [Info] Number of positive: 31074, number of negative: 31530
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.017588 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 64921
[LightGBM] [Info] Number of data points in the train set: 62604, number of used features: 267
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
[+] Model fitting complete.


In [3]:
print("\n" + "="*60)
print("             GLOBAL BASELINE SECURITY REPORT            ")
print("="*60)
print(classification_report(y_test, y_pred, target_names=["Benign", "Malicious"]))
print(f"Global ROC-AUC Score: {roc_auc_score(y_test, y_prob):.4f}")
print("="*60)

# --- TRACK METRICS PER STREAM (Crucial for Phase 5 verification) ---
print("\n[*] Evaluating performance per distinct stream category:")
test_df["pred"] = y_pred

for stream_name in test_df["stream"].unique():
    stream_mask = test_df["stream"] == stream_name
    sub_test = test_df[stream_mask]
    
    print(f"\nStream: {stream_name.upper()}")
    print(classification_report(sub_test["label"], sub_test["pred"], target_names=["Benign", "Malicious"], zero_division=0))


             GLOBAL BASELINE SECURITY REPORT            
              precision    recall  f1-score   support

      Benign       0.89      0.85      0.87      7894
   Malicious       0.85      0.90      0.87      7761

    accuracy                           0.87     15655
   macro avg       0.87      0.87      0.87     15655
weighted avg       0.87      0.87      0.87     15655

Global ROC-AUC Score: 0.9462

[*] Evaluating performance per distinct stream category:

Stream: REPO_FILE
              precision    recall  f1-score   support

      Benign       0.79      0.83      0.81       551
   Malicious       0.83      0.79      0.81       583

    accuracy                           0.81      1134
   macro avg       0.81      0.81      0.81      1134
weighted avg       0.81      0.81      0.81      1134


Stream: INDIRECT_CONTEXT
              precision    recall  f1-score   support

      Benign       0.90      0.84      0.87      7000
   Malicious       0.85      0.90      0.87    

/var/folders/cv/thg3bsp52jlbdy4ztqz6b6v40000gn/T/ipykernel_55435/2685492047.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test_df["pred"] = y_pred


<h1> Big models